In [2]:
from collections import defaultdict, deque

def distances_between_leaves(n, edges):
    # Build graph
    graph = defaultdict(list)
    for a, b, w in edges:
        graph[a].append((b, w))
        graph[b].append((a, w))

    # BFS to compute distances from a starting leaf
    def bfs(start):
        dist = {start: 0}
        queue = deque([start])
        while queue:
            u = queue.popleft()
            for v, w in graph[u]:
                if v not in dist:
                    dist[v] = dist[u] + w
                    queue.append(v)
        return dist

    # Compute n x n matrix
    result = [[0]*n for _ in range(n)]
    for i in range(n):
        dist = bfs(i)
        for j in range(n):
            result[i][j] = dist[j]

    return result


# -------------------------
# Example usage (sample input)
# -------------------------

n = 4
raw_edges = [
    "0->4:11", "1->4:2", "2->5:6", "3->5:7",
    "4->0:11", "4->1:2", "4->5:4",
    "5->4:4", "5->3:7", "5->2:6"
]

edges = []
for line in raw_edges:
    a, rest = line.split("->")
    b, w = rest.split(":")
    edges.append((int(a), int(b), int(w)))

matrix = distances_between_leaves(n, edges)

# Print space-separated matrix
for row in matrix:
    print(" ".join(map(str, row)))


0 13 21 22
13 0 12 13
21 12 0 13
22 13 13 0


In [11]:
data = """4
1
0 13 21 22
13 0 12 13
21 12 0 13
22 13 13 0
"""

# 把所有数字拆开
tokens = data.strip().split()

# 解析 n 和 j
n = int(tokens[0])
j = int(tokens[1])

# 剩下的是矩阵元素
nums = list(map(int, tokens[2:]))

# 构造矩阵 D
D = [nums[i*n:(i+1)*n] for i in range(n)]

def limb_length(n, j, D):
    limb = float('inf')
    for i in range(n):
        if i == j:
            continue
        for k in range(n):
            if k == j or k == i:
                continue
            value = (D[i][j] + D[j][k] - D[i][k]) // 2
            limb = min(limb, value)
    return limb

print(limb_length(n, j, D))

2


In [20]:
def limb_length(D, j):
    n = len(D)
    limb = float('inf')
    for i in range(n):
        if i == j:
            continue
        for k in range(n):
            if k == j or k == i:
                continue
            limb = min(limb, (D[i][j] + D[j][k] - D[i][k]) // 2)
    return limb


def find_attachment_point(tree, i, k, x):
    # DFS 找 i→k 的路径
    def dfs(u, parent, dist):
        if u == k:
            return [(u, dist)]
        for v, w in tree[u]:
            if v == parent:
                continue
            res = dfs(v, u, dist + w)
            if res:
                return [(u, dist)] + res
        return None

    path = dfs(i, -1, 0)

    # 在路径上找到距离 i 为 x 的点
    for idx in range(len(path) - 1):
        u, du = path[idx]
        v, dv = path[idx + 1]
        if du <= x <= dv:
            if x == du:
                return u
            if x == dv:
                return v

            # 在边中间插入新节点
            new_node = max(tree.keys()) + 1
            w = dv - du

            tree[u].remove((v, w))
            tree[v].remove((u, w))

            tree[u].append((new_node, x - du))
            tree[new_node] = [(u, x - du)]
            tree[v].append((new_node, dv - x))
            tree[new_node].append((v, dv - x))

            return new_node

    return None


def additive_phylogeny(D):
    n = len(D)
    if n == 2:
        return {0: [(1, D[0][1])], 1: [(0, D[0][1])]}

    j = n - 1
    limb = limb_length(D, j)

    # 保存原始距离
    D_orig = [row[:] for row in D]

    # 修剪 limb
    for i in range(n):
        if i != j:
            D[i][j] -= limb
            D[j][i] = D[i][j]

    # 找 i,k
    i = k = None
    for a in range(n):
        for b in range(n):
            if a != j and b != j and D[a][b] == D[a][j] + D[j][b]:
                i, k = a, b
                break
        if i is not None:
            break

    # 正确的 x
    x = D_orig[i][j] - limb

    # 删除 j 行列
    D_trim = [row[:j] + row[j+1:] for row in D[:j] + D[j+1:]]

    # 递归
    tree = additive_phylogeny(D_trim)

    # 找挂载点
    v = find_attachment_point(tree, i, k, x)

    # 接回 j
    tree[j] = []
    tree[v].append((j, limb))
    tree[j].append((v, limb))

    return tree


def solve(data):
    tokens = data.strip().split()
    n = int(tokens[0])
    nums = list(map(int, tokens[1:]))
    D = [nums[i*n:(i+1)*n] for i in range(n)]

    tree = additive_phylogeny(D)

    result = []
    for u in sorted(tree.keys()):
        for v, w in tree[u]:
            result.append(f"{u}->{v}:{w}")
    return "\n".join(result)
